# Encoder LR Sweep — Paper 2: Semantic Drift, Not Rank Collapse

**Question:** Is the free encoder's drift a tuning problem (wrong LR) or a structural one?

**Design:** Free encoder at LR = {1e-4, 3e-4, 1e-3, 3e-3} vs prescribed (LR irrelevant).
Seed=42, 50 epochs. Covariance eigenvalue tracking + R² transfer every epoch.

**Expected result:** Prescribed wins at every LR. Drift metrics (eigenvalue instability, R² < 0)
persist regardless of LR — confirming drift is structural, not a tuning artifact.

In [ ]:
!pip install pymunk==6.6.0 gym-pusht torch numpy scipy scikit-learn matplotlib -q

In [ ]:
# ============================================================
# Google Drive: mount + resume
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/lr_sweep_paper2'
import os
os.makedirs(DRIVE_DIR, exist_ok=True)

# Resume: load previously saved results if they exist
import json as _json
RESULTS_PATH = os.path.join(DRIVE_DIR, 'lr_sweep_results.json')
COMPLETED_PATH = os.path.join(DRIVE_DIR, 'completed_conditions.json')

completed_conditions = []
resumed_results = {}
if os.path.exists(COMPLETED_PATH):
    with open(COMPLETED_PATH) as f:
        completed_conditions = _json.load(f)
    print(f"Resume: {len(completed_conditions)} conditions already done: {completed_conditions}")
if os.path.exists(RESULTS_PATH):
    with open(RESULTS_PATH) as f:
        resumed_results = _json.load(f)
    print(f"Loaded {len(resumed_results)} saved results")
else:
    print("Fresh start — no previous results found")

def save_to_drive(results_dict, condition_name):
    """Save results + mark condition as completed after each run."""
    # Save full results
    save_data = {}
    for k, v in results_dict.items():
        save_data[k] = {
            'label': v['label'], 'mode': v['mode'], 'lr': v['lr'],
            'best_vp': v['best_vp'], 'best_ep': v['best_ep'], 'n_params': v['n_params'],
            'history': v['history'],
            'cov_history': v['cov_history'],
            'r2_history': v['r2_history'],
        }
        if 'ema_decay' in v:
            save_data[k]['ema_decay'] = v['ema_decay']
    with open(RESULTS_PATH, 'w') as f:
        _json.dump(save_data, f, indent=2)
    # Mark completed
    if condition_name not in completed_conditions:
        completed_conditions.append(condition_name)
    with open(COMPLETED_PATH, 'w') as f:
        _json.dump(completed_conditions, f)
    print(f"  >> Saved to Drive: {condition_name} ({len(completed_conditions)} total)")

In [ ]:
import os, time, json, copy
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from scipy.linalg import orthogonal_procrustes
from sklearn.linear_model import LinearRegression
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# ============================================================
# SIGReg (from LeWM module.py)
# ============================================================
class SIGReg(nn.Module):
    def __init__(self, knots=17, num_proj=256):
        super().__init__()
        self.num_proj = num_proj
        t = torch.linspace(0, 3, knots)
        dt = 3/(knots-1)
        w = torch.full((knots,), 2*dt); w[[0,-1]] = dt
        phi = torch.exp(-t.square()/2.0)
        self.register_buffer("t", t)
        self.register_buffer("phi", phi)
        self.register_buffer("weights", w*phi)

    def forward(self, proj):
        A = torch.randn(proj.size(-1), self.num_proj, device=proj.device)
        A = A.div_(A.norm(p=2, dim=0))
        x_t = (proj @ A).unsqueeze(-1) * self.t
        err = (x_t.cos().mean(-3)-self.phi).square() + x_t.sin().mean(-3).square()
        return ((err @ self.weights)*proj.size(-2)).mean()


# ============================================================
# Data collection
# ============================================================
def collect_gym_data(n_ep=200, max_steps=300, fs=5, seed=42):
    try:
        import gymnasium as gym
        import gym_pusht
    except ImportError:
        print("gym-pusht not found -> synthetic")
        return collect_synthetic_data(n_ep, max_steps, fs, seed)
    rng = np.random.default_rng(seed)
    eps = []
    env = gym.make("gym_pusht/PushT-v0", obs_type="state", render_mode=None)
    for i in range(n_ep):
        obs, _ = env.reset(seed=int(rng.integers(0,100000)))
        ss, aa = [obs.copy()], []
        for step in range(max_steps):
            if step == 0 or rng.random() < 0.3:
                a = env.action_space.sample()
            else:
                a = aa[-1] + rng.normal(0, 30, 2).astype(np.float32)
                a = np.clip(a, 0, 512)
            obs, _, d, tr, _ = env.step(a)
            ss.append(obs.copy())
            aa.append(a.copy())
            if d or tr:
                break
        ss, aa = np.array(ss), np.array(aa)
        for j in range(0, len(aa) - fs, fs):
            eps.append((ss[j:j+fs+1], aa[j:j+fs]))
        if (i+1) % 50 == 0:
            print(f"  collected {i+1}/{n_ep} episodes, {len(eps)} windows")
    env.close()
    return eps

def collect_synthetic_data(n_ep=200, max_steps=300, fs=5, seed=42):
    rng = np.random.default_rng(seed)
    eps = []
    for i in range(n_ep):
        state = rng.uniform([0,0,0,0,0], [512,512,512,512,2*np.pi])
        ss, aa = [state.copy()], []
        for step in range(max_steps):
            a = rng.uniform(0, 512, 2).astype(np.float32)
            dx = (a - state[:2]) * 0.1
            state[:2] += dx
            state[2] += (a[0] - state[2]) * 0.05
            state[3] += (a[1] - state[3]) * 0.05
            state[4] += 0.02 * np.sin(a[0] / 100)
            state = np.clip(state, [0,0,0,0,0], [512,512,512,512,2*np.pi])
            ss.append(state.copy())
            aa.append(a.copy())
        ss, aa = np.array(ss), np.array(aa)
        for j in range(0, len(aa) - fs, fs):
            eps.append((ss[j:j+fs+1], aa[j:j+fs]))
    return eps


class PushTDataset(Dataset):
    def __init__(self, windows):
        self.s = [torch.tensor(s, dtype=torch.float32) for s, a in windows]
        self.a = [torch.tensor(a, dtype=torch.float32) for s, a in windows]

    def __len__(self):
        return len(self.s)

    def __getitem__(self, i):
        return self.s[i], self.a[i]


print("Collecting data...")
windows = collect_gym_data(200, 300, 5, 42)
ds = PushTDataset(windows)
n_val = max(1, len(ds) // 5)
train_ds, val_ds = random_split(ds, [len(ds) - n_val, n_val],
                                 generator=torch.Generator().manual_seed(42))
train_dl = DataLoader(train_ds, batch_size=128, shuffle=True, drop_last=True)
val_dl = DataLoader(val_ds, batch_size=256)
print(f"Train: {len(train_ds)}, Val: {len(val_ds)}")

In [ ]:
# ============================================================
# Encoders
# ============================================================
def make_prescribed_encoder(state_dim, D=3):
    scale = torch.tensor([1.0/512, 1.0/512, 1.0/(2*np.pi)])
    def encode(s):
        return s[..., 2:5] * scale.to(s.device)
    return encode, []

def make_free_encoder(state_dim, D=3):
    enc = nn.Sequential(
        nn.Linear(state_dim, 128), nn.LayerNorm(128), nn.GELU(),
        nn.Linear(128, 128), nn.LayerNorm(128), nn.GELU(),
        nn.Linear(128, D),
    )
    return enc, list(enc.parameters())

def make_predictor(D=3, H=4, act_dim=2):
    aenc = nn.Sequential(nn.Linear(act_dim, 32), nn.GELU(), nn.Linear(32, D))
    pred = nn.Sequential(
        nn.Linear(H * D * 2, 128), nn.GELU(),
        nn.Linear(128, 128), nn.GELU(),
        nn.Linear(128, D),
    )
    params = list(aenc.parameters()) + list(pred.parameters())
    return aenc, pred, params


# ============================================================
# Covariance tracking
# ============================================================
def compute_cov_eigenvalues(enc, dl, is_prescribed=False):
    """Compute eigenvalues of embedding covariance matrix."""
    all_emb = []
    with torch.no_grad():
        for s, a in dl:
            s = s.to(device)
            if is_prescribed:
                emb = enc(s[:, 0])
            else:
                enc.eval()
                emb = enc(s[:, 0])
            all_emb.append(emb.cpu())
    emb = torch.cat(all_emb, 0)
    emb_np = emb.numpy()
    cov = np.cov(emb_np.T)
    eigenvalues = np.sort(np.linalg.eigvalsh(cov))[::-1]
    return eigenvalues


def compute_r2_transfer(emb_prev, emb_curr):
    """R² of linear regression: can epoch N embeddings predict epoch N+1?"""
    if emb_prev is None:
        return None
    reg = LinearRegression().fit(emb_prev, emb_curr)
    return reg.score(emb_prev, emb_curr)


def collect_embeddings(enc, dl, is_prescribed=False):
    all_emb = []
    with torch.no_grad():
        for s, a in dl:
            s = s.to(device)
            if is_prescribed:
                emb = enc(s[:, 0])
            else:
                enc.eval()
                emb = enc(s[:, 0])
            all_emb.append(emb.cpu().numpy())
    return np.concatenate(all_emb, 0)

In [ ]:
def run_experiment_with_tracking(mode, train_dl, val_dl, D=3, H=4,
                                  epochs=50, lr=3e-4, lam=0.05, seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)

    state_dim = 5
    for s, a in train_dl:
        state_dim = s.shape[-1]
        break

    is_prescribed = (mode == 'prescribed')

    if is_prescribed:
        enc, enc_params = make_prescribed_encoder(state_dim, D)
    else:
        enc, enc_params = make_free_encoder(state_dim, D)
        enc = enc.to(device)

    aenc, pred, pred_params = make_predictor(D, H)
    aenc, pred = aenc.to(device), pred.to(device)
    sigreg = SIGReg(knots=17, num_proj=256).to(device)

    all_params = enc_params + pred_params
    n_params = sum(p.numel() for p in all_params)
    opt = torch.optim.AdamW(all_params, lr=lr, weight_decay=1e-3) if all_params else None

    label = f"{mode}_lr{lr}"
    print(f"\n{'='*60}")
    print(f"  {label.upper()} ({n_params:,} params)")
    print(f"{'='*60}")

    history = []
    cov_history = []  # eigenvalues per epoch
    r2_history = []   # R² transfer per epoch
    prev_emb = None
    best_vp, best_ep = float('inf'), 0

    for ep in range(1, epochs + 1):
        t0 = time.time()

        # --- Collect embeddings before training this epoch ---
        curr_emb = collect_embeddings(enc, val_dl, is_prescribed)
        eigvals = compute_cov_eigenvalues(enc, val_dl, is_prescribed)
        r2 = compute_r2_transfer(prev_emb, curr_emb)
        prev_emb = curr_emb.copy()
        cov_history.append(eigvals.tolist())
        r2_history.append(r2)

        # --- Train ---
        if not is_prescribed:
            enc.train()
        aenc.train(); pred.train()
        tp, ts, n = 0, 0, 0
        for s, a in train_dl:
            s, a = s.to(device), a.to(device)
            # Encode
            embs = []
            for t in range(s.size(1)):
                embs.append(enc(s[:, t]))
            embs = torch.stack(embs, 1)
            # Action encoding
            ae = []
            for t in range(a.size(1)):
                ae.append(aenc(a[:, t]))
            ae = torch.stack(ae, 1)
            # Predict
            ctx = embs[:, :H]
            tgt = embs[:, H:]
            losses_p = []
            for t in range(tgt.size(1)):
                inp = torch.cat([ctx, ae[:, :H+t]], dim=1) if t > 0 else ctx
                inp = torch.cat([inp[:, -H:], ae[:, t:t+1].expand(-1, H, -1)], dim=-1)
                inp = inp.reshape(inp.size(0), -1)
                p = pred(inp)
                losses_p.append(F.mse_loss(p, tgt[:, t]))
            lp = torch.stack(losses_p).mean()
            ls = sigreg(embs.reshape(-1, D)) * lam if not is_prescribed else torch.tensor(0.0)
            loss = lp + ls
            if opt:
                opt.zero_grad()
                loss.backward()
                opt.step()
            tp += lp.item() * s.size(0)
            ts += ls.item() * s.size(0) if not is_prescribed else 0
            n += s.size(0)

        # --- Val ---
        if not is_prescribed:
            enc.eval()
        aenc.eval(); pred.eval()
        vp, vn = 0, 0
        with torch.no_grad():
            for s, a in val_dl:
                s, a = s.to(device), a.to(device)
                embs = []
                for t in range(s.size(1)):
                    embs.append(enc(s[:, t]))
                embs = torch.stack(embs, 1)
                ae = []
                for t in range(a.size(1)):
                    ae.append(aenc(a[:, t]))
                ae = torch.stack(ae, 1)
                ctx = embs[:, :H]
                tgt = embs[:, H:]
                for t in range(tgt.size(1)):
                    inp = torch.cat([ctx, ae[:, :H+t]], dim=1) if t > 0 else ctx
                    inp = torch.cat([inp[:, -H:], ae[:, t:t+1].expand(-1, H, -1)], dim=-1)
                    inp = inp.reshape(inp.size(0), -1)
                    p = pred(inp)
                    vp += F.mse_loss(p, tgt[:, t]).item() * s.size(0)
                vn += s.size(0)

        val_loss = vp / max(vn, 1)
        if val_loss < best_vp:
            best_vp = val_loss
            best_ep = ep

        dt = time.time() - t0
        r2_str = f"{r2:.4f}" if r2 is not None else "N/A"
        if ep % 10 == 0 or ep == 1:
            print(f"  ep {ep:3d} | train_p={tp/n:.6f} sig={ts/n:.6f} | val={val_loss:.6f} | best={best_vp:.6f}@{best_ep} | R²={r2_str} | {dt:.1f}s")

        history.append({
            'ep': ep, 'train_pred': tp/n, 'train_sig': ts/n,
            'val_pred': val_loss, 'best_vp': best_vp, 'best_ep': best_ep
        })

    return {
        'label': label, 'mode': mode, 'lr': lr,
        'best_vp': best_vp, 'best_ep': best_ep, 'n_params': n_params,
        'history': history, 'cov_history': cov_history, 'r2_history': r2_history
    }

In [ ]:
# ============================================================
# ENCODER LR SWEEP (with resume + Drive save)
# ============================================================
LR_VALUES = [1e-4, 3e-4, 1e-3, 3e-3]
SEED = 42

results = {}

# Restore previously completed results
for k, v in resumed_results.items():
    results[k] = v
    # Add 'lr' field if missing (for plotting)
    if 'lr' not in v:
        v['lr'] = 3e-4

# Prescribed baseline
if 'prescribed' in completed_conditions:
    print(f"SKIP prescribed (already done): best_vp={results['prescribed']['best_vp']:.6f}")
else:
    print("=" * 70)
    print("  PRESCRIBED BASELINE")
    print("=" * 70)
    r = run_experiment_with_tracking('prescribed', train_dl, val_dl,
                                      lr=3e-4, seed=SEED)
    results['prescribed'] = r
    save_to_drive(results, 'prescribed')
    print(f"\n  PRESCRIBED: best_vp={r['best_vp']:.6f} @ ep {r['best_ep']}")

# Free at each LR
for lr_val in LR_VALUES:
    key = f'free_lr{lr_val}'
    if key in completed_conditions:
        print(f"SKIP {key} (already done): best_vp={results[key]['best_vp']:.6f}")
        continue
    print(f"\n{'=' * 70}")
    print(f"  FREE ENCODER LR={lr_val}")
    print(f"{'=' * 70}")
    r = run_experiment_with_tracking('free', train_dl, val_dl,
                                      lr=lr_val, seed=SEED)
    results[key] = r
    save_to_drive(results, key)
    print(f"\n  FREE LR={lr_val}: best_vp={r['best_vp']:.6f} @ ep {r['best_ep']}")

# Summary
print("\n" + "=" * 70)
print("  SUMMARY")
print("=" * 70)
print(f"  {'Condition':<25s} {'Best Val Loss':>12s} {'Best Ep':>8s} {'Params':>8s}")
print(f"  {'-'*25} {'-'*12} {'-'*8} {'-'*8}")
for k, v in results.items():
    print(f"  {k:<25s} {v['best_vp']:>12.6f} {v['best_ep']:>8d} {v['n_params']:>8,d}")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# --- Plot 1: Val loss curves ---
ax = axes[0, 0]
r = results['prescribed']
ax.plot([h['val_pred'] for h in r['history']], 'k-', linewidth=2, label='prescribed')
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
for i, lr_val in enumerate(LR_VALUES):
    r = results[f'free_lr{lr_val}']
    ax.plot([h['val_pred'] for h in r['history']], color=colors[i],
            label=f'free LR={lr_val}')
ax.set_yscale('log')
ax.set_xlabel('Epoch')
ax.set_ylabel('Val Loss (log)')
ax.set_title('Prediction Loss vs Epoch')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# --- Plot 2: Covariance eigenvalues over time ---
ax = axes[0, 1]
# For each free condition, plot max eigenvalue / min eigenvalue ratio
for i, lr_val in enumerate(LR_VALUES):
    r = results[f'free_lr{lr_val}']
    ratios = []
    for eigvals in r['cov_history']:
        eigvals = np.array(eigvals)
        ratio = eigvals[0] / max(eigvals[-1], 1e-12)
        ratios.append(ratio)
    ax.plot(ratios, color=colors[i], label=f'free LR={lr_val}')
r = results['prescribed']
ratios_p = []
for eigvals in r['cov_history']:
    eigvals = np.array(eigvals)
    ratio = eigvals[0] / max(eigvals[-1], 1e-12)
    ratios_p.append(ratio)
ax.plot(ratios_p, 'k-', linewidth=2, label='prescribed')
ax.set_xlabel('Epoch')
ax.set_ylabel('Condition Number (λ_max / λ_min)')
ax.set_title('Covariance Condition Number')
ax.set_yscale('log')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# --- Plot 3: R² transfer ---
ax = axes[1, 0]
for i, lr_val in enumerate(LR_VALUES):
    r = results[f'free_lr{lr_val}']
    r2_vals = [v if v is not None else np.nan for v in r['r2_history']]
    ax.plot(r2_vals, color=colors[i], label=f'free LR={lr_val}')
r = results['prescribed']
r2_vals_p = [v if v is not None else np.nan for v in r['r2_history']]
ax.plot(r2_vals_p, 'k-', linewidth=2, label='prescribed')
ax.axhline(y=0, color='red', linestyle='--', alpha=0.5, label='R²=0 (no transfer)')
ax.set_xlabel('Epoch')
ax.set_ylabel('R² (linear transfer)')
ax.set_title('R² Transfer Between Consecutive Epochs')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# --- Plot 4: Eigenvalue spectra at epoch 1, 25, 50 ---
ax = axes[1, 1]
epochs_to_show = [0, 24, 49]
epoch_labels = ['ep 1', 'ep 25', 'ep 50']
# Show prescribed + best free + worst free
best_free_key = min([k for k in results if k.startswith('free')],
                     key=lambda k: results[k]['best_vp'])
worst_free_key = max([k for k in results if k.startswith('free')],
                      key=lambda k: results[k]['best_vp'])
for idx, ep_idx in enumerate(epochs_to_show):
    offset = idx * 0.25
    eigvals_p = np.array(results['prescribed']['cov_history'][ep_idx])
    eigvals_best = np.array(results[best_free_key]['cov_history'][ep_idx])
    eigvals_worst = np.array(results[worst_free_key]['cov_history'][ep_idx])
    x = np.arange(len(eigvals_p))
    ax.bar(x + offset - 0.08, eigvals_p, 0.08, color='black', alpha=0.7,
           label=f'prescribed {epoch_labels[idx]}' if idx == 0 else '')
    ax.bar(x + offset, eigvals_best, 0.08, color='blue', alpha=0.5,
           label=f'{best_free_key} {epoch_labels[idx]}' if idx == 0 else '')
    ax.bar(x + offset + 0.08, eigvals_worst, 0.08, color='red', alpha=0.5,
           label=f'{worst_free_key} {epoch_labels[idx]}' if idx == 0 else '')
ax.set_xlabel('Eigenvalue Index')
ax.set_ylabel('Eigenvalue')
ax.set_title('Eigenvalue Spectra at Different Epochs')
ax.legend(fontsize=7)
ax.grid(True, alpha=0.3)

plt.suptitle('Encoder LR Sweep — Prescribed vs Free at Different Learning Rates',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('lr_sweep_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: lr_sweep_results.png')

In [ ]:
# Save full results as JSON
save_data = {}
for k, v in results.items():
    save_data[k] = {
        'label': v['label'], 'mode': v['mode'], 'lr': v['lr'],
        'best_vp': v['best_vp'], 'best_ep': v['best_ep'], 'n_params': v['n_params'],
        'history': v['history'],
        'cov_history': v['cov_history'],
        'r2_history': v['r2_history'],
    }

with open('lr_sweep_results.json', 'w') as f:
    json.dump(save_data, f, indent=2)

print("Results saved to lr_sweep_results.json")
print("\nKey findings:")
prescribed_vp = results['prescribed']['best_vp']
for lr_val in LR_VALUES:
    free_vp = results[f'free_lr{lr_val}']['best_vp']
    ratio = free_vp / prescribed_vp if prescribed_vp > 0 else float('inf')
    r2_ep1 = results[f'free_lr{lr_val}']['r2_history'][1]
    r2_str = f"{r2_ep1:.4f}" if r2_ep1 is not None else "N/A"
    print(f"  LR={lr_val}: free/prescribed = {ratio:.1f}x, R²(ep0→1) = {r2_str}")

## EMA Baseline

Standard JEPA mechanism: EMA target encoder provides stable targets.
Question: does EMA eliminate drift, or just slow it down?

In [ ]:
def run_ema_experiment(train_dl, val_dl, D=3, H=4, epochs=50,
                       lr=3e-4, lam=0.05, ema_decay=0.996, seed=42):
    """Free encoder with EMA target encoder (standard I-JEPA mechanism)."""
    torch.manual_seed(seed)
    np.random.seed(seed)

    state_dim = 5
    for s, a in train_dl:
        state_dim = s.shape[-1]
        break

    enc, enc_params = make_free_encoder(state_dim, D)
    enc = enc.to(device)
    # EMA copy
    ema_enc = copy.deepcopy(enc)
    for p in ema_enc.parameters():
        p.requires_grad = False

    aenc, pred, pred_params = make_predictor(D, H)
    aenc, pred = aenc.to(device), pred.to(device)
    sigreg = SIGReg(knots=17, num_proj=256).to(device)

    all_params = enc_params + pred_params
    n_params = sum(p.numel() for p in all_params)
    opt = torch.optim.AdamW(all_params, lr=lr, weight_decay=1e-3)

    print(f"\n{'='*60}")
    print(f"  FREE + EMA (decay={ema_decay}, {n_params:,} params)")
    print(f"{'='*60}")

    history = []
    cov_history = []
    r2_history = []
    prev_emb = None
    best_vp, best_ep = float('inf'), 0

    for ep in range(1, epochs + 1):
        t0 = time.time()

        # Collect embeddings (from online encoder)
        curr_emb = collect_embeddings(enc, val_dl, False)
        eigvals = compute_cov_eigenvalues(enc, val_dl, False)
        r2 = compute_r2_transfer(prev_emb, curr_emb)
        prev_emb = curr_emb.copy()
        cov_history.append(eigvals.tolist())
        r2_history.append(r2)

        # Train
        enc.train(); aenc.train(); pred.train()
        tp, ts, n = 0, 0, 0
        for s, a in train_dl:
            s, a = s.to(device), a.to(device)
            # Online encoder for context
            embs_online = []
            for t in range(s.size(1)):
                embs_online.append(enc(s[:, t]))
            embs_online = torch.stack(embs_online, 1)
            # EMA encoder for targets (with stop-gradient)
            with torch.no_grad():
                embs_ema = []
                for t in range(s.size(1)):
                    embs_ema.append(ema_enc(s[:, t]))
                embs_ema = torch.stack(embs_ema, 1)
            # Action encoding
            ae = []
            for t in range(a.size(1)):
                ae.append(aenc(a[:, t]))
            ae = torch.stack(ae, 1)
            # Predict: context from online, targets from EMA
            ctx = embs_online[:, :H]
            tgt = embs_ema[:, H:]  # EMA targets
            losses_p = []
            for t in range(tgt.size(1)):
                inp = torch.cat([ctx, ae[:, :H+t]], dim=1) if t > 0 else ctx
                inp = torch.cat([inp[:, -H:], ae[:, t:t+1].expand(-1, H, -1)], dim=-1)
                inp = inp.reshape(inp.size(0), -1)
                p = pred(inp)
                losses_p.append(F.mse_loss(p, tgt[:, t].detach()))
            lp = torch.stack(losses_p).mean()
            ls = sigreg(embs_online.reshape(-1, D)) * lam
            loss = lp + ls
            opt.zero_grad()
            loss.backward()
            opt.step()
            # EMA update
            with torch.no_grad():
                for p_online, p_ema in zip(enc.parameters(), ema_enc.parameters()):
                    p_ema.data.mul_(ema_decay).add_(p_online.data, alpha=1 - ema_decay)
            tp += lp.item() * s.size(0)
            ts += ls.item() * s.size(0)
            n += s.size(0)

        # Val (use online encoder for consistency)
        enc.eval(); aenc.eval(); pred.eval()
        vp, vn = 0, 0
        with torch.no_grad():
            for s, a in val_dl:
                s, a = s.to(device), a.to(device)
                embs = []
                for t in range(s.size(1)):
                    embs.append(enc(s[:, t]))
                embs = torch.stack(embs, 1)
                ae = []
                for t in range(a.size(1)):
                    ae.append(aenc(a[:, t]))
                ae = torch.stack(ae, 1)
                ctx = embs[:, :H]
                tgt = embs[:, H:]
                for t in range(tgt.size(1)):
                    inp = torch.cat([ctx, ae[:, :H+t]], dim=1) if t > 0 else ctx
                    inp = torch.cat([inp[:, -H:], ae[:, t:t+1].expand(-1, H, -1)], dim=-1)
                    inp = inp.reshape(inp.size(0), -1)
                    p = pred(inp)
                    vp += F.mse_loss(p, tgt[:, t]).item() * s.size(0)
                vn += s.size(0)

        val_loss = vp / max(vn, 1)
        if val_loss < best_vp:
            best_vp = val_loss
            best_ep = ep

        dt = time.time() - t0
        r2_str = f"{r2:.4f}" if r2 is not None else "N/A"
        if ep % 10 == 0 or ep == 1:
            print(f"  ep {ep:3d} | train_p={tp/n:.6f} sig={ts/n:.6f} | val={val_loss:.6f} | best={best_vp:.6f}@{best_ep} | R²={r2_str} | {dt:.1f}s")

        history.append({
            'ep': ep, 'train_pred': tp/n, 'train_sig': ts/n,
            'val_pred': val_loss, 'best_vp': best_vp, 'best_ep': best_ep
        })

    return {
        'label': f'free_ema_{ema_decay}', 'mode': 'free_ema', 'lr': lr,
        'ema_decay': ema_decay,
        'best_vp': best_vp, 'best_ep': best_ep, 'n_params': n_params,
        'history': history, 'cov_history': cov_history, 'r2_history': r2_history
    }

# Run EMA baseline
ema_result = run_ema_experiment(train_dl, val_dl, seed=42)
results['free_ema'] = ema_result
print(f"\nEMA BASELINE: best_vp={ema_result['best_vp']:.6f} @ ep {ema_result['best_ep']}")
print(f"PRESCRIBED:   best_vp={results['prescribed']['best_vp']:.6f}")
print(f"FREE (3e-4):  best_vp={results['free_lr0.0003']['best_vp']:.6f}")

# Check if already done
if 'free_ema' in completed_conditions:
    results['free_ema'] = resumed_results.get('free_ema', {})
    ema_result = results['free_ema']
    print(f"SKIP free_ema (already done): best_vp={ema_result['best_vp']:.6f}")
else:
    # Run EMA baseline
    ema_result = run_ema_experiment(train_dl, val_dl, seed=42)
    results['free_ema'] = ema_result
    save_to_drive(results, 'free_ema')

print(f"\nEMA BASELINE: best_vp={results['free_ema']['best_vp']:.6f}")
print(f"PRESCRIBED:   best_vp={results['prescribed']['best_vp']:.6f}")
free_key = 'free_lr0.0003'
if free_key in results:
    print(f"FREE (3e-4):  best_vp={results[free_key]['best_vp']:.6f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Val loss: prescribed vs free(3e-4) vs EMA
ax = axes[0]
ax.plot([h['val_pred'] for h in results['prescribed']['history']],
        'k-', linewidth=2, label='prescribed')
ax.plot([h['val_pred'] for h in results['free_lr0.0003']['history']],
        'b-', label='free (LR=3e-4)')
ax.plot([h['val_pred'] for h in results['free_ema']['history']],
        'r-', label='free + EMA')
ax.set_yscale('log')
ax.set_xlabel('Epoch')
ax.set_ylabel('Val Loss (log)')
ax.set_title('Val Loss: Prescribed vs Free vs EMA')
ax.legend()
ax.grid(True, alpha=0.3)

# Condition number
ax = axes[1]
for key, color, label in [('prescribed', 'black', 'prescribed'),
                            ('free_lr0.0003', 'blue', 'free'),
                            ('free_ema', 'red', 'free+EMA')]:
    ratios = []
    for eigvals in results[key]['cov_history']:
        eigvals = np.array(eigvals)
        ratio = eigvals[0] / max(eigvals[-1], 1e-12)
        ratios.append(ratio)
    ax.plot(ratios, color=color, linewidth=2 if key == 'prescribed' else 1, label=label)
ax.set_yscale('log')
ax.set_xlabel('Epoch')
ax.set_ylabel('Condition Number')
ax.set_title('Covariance Condition Number')
ax.legend()
ax.grid(True, alpha=0.3)

# R² transfer
ax = axes[2]
for key, color, label in [('prescribed', 'black', 'prescribed'),
                            ('free_lr0.0003', 'blue', 'free'),
                            ('free_ema', 'red', 'free+EMA')]:
    r2_vals = [v if v is not None else np.nan for v in results[key]['r2_history']]
    ax.plot(r2_vals, color=color, linewidth=2 if key == 'prescribed' else 1, label=label)
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Epoch')
ax.set_ylabel('R²')
ax.set_title('R² Transfer Between Epochs')
ax.legend()
ax.grid(True, alpha=0.3)

plt.suptitle('LR Sweep + EMA Baseline — Drift is Structural, Not Tuning',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('lr_sweep_ema_combined.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: lr_sweep_ema_combined.png')

In [ ]:
# Add EMA to saved results
save_data['free_ema'] = {
    'label': ema_result['label'], 'mode': ema_result['mode'],
    'lr': ema_result['lr'], 'ema_decay': ema_result['ema_decay'],
    'best_vp': ema_result['best_vp'], 'best_ep': ema_result['best_ep'],
    'n_params': ema_result['n_params'],
    'history': ema_result['history'],
    'cov_history': ema_result['cov_history'],
    'r2_history': ema_result['r2_history'],
}

with open('lr_sweep_results.json', 'w') as f:
    json.dump(save_data, f, indent=2)

print("\n" + "="*70)
print("FINAL SUMMARY")
print("="*70)
prescribed_vp = results['prescribed']['best_vp']
print(f"  {'Condition':<25s} {'Best Val':>10s} {'Ratio':>8s} {'R²(0→1)':>10s}")
print(f"  {'-'*25} {'-'*10} {'-'*8} {'-'*10}")
print(f"  {'prescribed':<25s} {prescribed_vp:>10.6f} {'1.0x':>8s} {'1.0000':>10s}")
for k in sorted([k for k in results if k != 'prescribed']):
    v = results[k]
    ratio = v['best_vp'] / prescribed_vp if prescribed_vp > 0 else float('inf')
    r2_1 = v['r2_history'][1] if len(v['r2_history']) > 1 and v['r2_history'][1] is not None else None
    r2_str = f"{r2_1:.4f}" if r2_1 is not None else "N/A"
    print(f"  {k:<25s} {v['best_vp']:>10.6f} {ratio:>7.1f}x {r2_str:>10s}")

print("\nDownload: lr_sweep_results.json, lr_sweep_results.png, lr_sweep_ema_combined.png")